# W&B Run Download And Local Plots

This notebook downloads selected W&B run histories, caches them locally, and plots training/evaluation curves without depending on the W&B UI.

Before running it, make sure you are logged in:

```bash
wandb login
```

The downloaded cache is written to `Topology_Task/outputs/wandb_cache/` and figures are written to `Topology_Task/outputs/wandb_figures/`.

In [ ]:
from pathlib import Path
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import wandb
except ImportError as exc:
    raise ImportError("Install wandb first, for example: pip install wandb") from exc

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)


## Configuration

Edit this cell to select runs. The default regex targets the 8 new GINE runs.

In [ ]:
ENTITY = os.getenv("WANDB_ENTITY", "corentin-plumet-epfl")
PROJECT = os.getenv("WANDB_PROJECT", "Grid2Op")

# Examples:
#   r"^gine20_"                  -> the 8 new GINE configs
#   r"noval20_mlp|split90_mlp"   -> recent split/no-validation MLP runs
#   r"a3_40env|a1_p0_03"         -> older interesting runs
RUN_NAME_REGEX = r"^gine20_"
RUN_STATES = None  # None, or for example {"finished", "running"}
MAX_RUNS = None    # None means all matching runs

FORCE_REFRESH = False
PAGE_SIZE = 1000

METRICS = [
    "charts/episodic_survival",
    "test/charts/episodic_survival",
    "test/episodic_survival",
    "train_eval/charts/episodic_survival",
    "train_eval/episodic_survival",
    "train/entropy_coef",
    "train/action0_logit_bonus",
    "train/entropy_agent_0",
    "train/entropy_agent_1",
    "train/entropy_agent_2",
    "train/frac_action_0_agent_0",
    "train/frac_action_0_agent_1",
    "train/frac_action_0_agent_2",
    "train/approx_kl_agent_0",
    "train/approx_kl_agent_1",
    "train/approx_kl_agent_2",
    "train/lr_actor",
    "train/lr_critic",
]

cwd = Path.cwd().resolve()
if (cwd / "main.py").is_file():
    TASK_DIR = cwd
elif (cwd / "Topology_Task" / "main.py").is_file():
    TASK_DIR = cwd / "Topology_Task"
elif (cwd.parent / "main.py").is_file():
    TASK_DIR = cwd.parent
else:
    raise RuntimeError("Could not locate Topology_Task/main.py from the current working directory.")

CACHE_DIR = TASK_DIR / "outputs" / "wandb_cache"
FIG_DIR = TASK_DIR / "outputs" / "wandb_figures"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project: {ENTITY}/{PROJECT}")
print(f"Task dir: {TASK_DIR}")
print(f"Cache dir: {CACHE_DIR}")


## Fetch Run List

In [ ]:
api = wandb.Api(timeout=60)
all_runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
name_re = re.compile(RUN_NAME_REGEX) if RUN_NAME_REGEX else None

def run_matches(run):
    if RUN_STATES is not None and run.state not in RUN_STATES:
        return False
    if name_re is None:
        return True
    exp_tag = str(run.config.get("exp_tag", ""))
    return bool(name_re.search(run.name) or name_re.search(exp_tag))

selected_runs = [run for run in all_runs if run_matches(run)]
selected_runs = sorted(selected_runs, key=lambda run: getattr(run, "created_at", "") or "")
if MAX_RUNS is not None:
    selected_runs = selected_runs[:MAX_RUNS]

summary_rows = []
for run in selected_runs:
    summary = dict(run.summary)
    config = dict(run.config)
    summary_rows.append({
        "name": run.name,
        "id": run.id,
        "state": run.state,
        "created_at": getattr(run, "created_at", None),
        "exp_tag": config.get("exp_tag"),
        "actor_encoder": config.get("actor_encoder"),
        "critic_encoder": config.get("critic_encoder"),
        "gnn_type": config.get("gnn_type"),
        "deterministic_eval": config.get("deterministic_eval"),
        "n_envs": config.get("n_envs"),
        "n_steps": config.get("n_steps"),
        "entropy_coef": config.get("entropy_coef"),
        "entropy_coef_final": config.get("entropy_coef_final"),
        "init_do_nothing_prob": config.get("init_do_nothing_prob"),
        "global_step": summary.get("charts/global_step"),
        "test_survival": summary.get("test/charts/episodic_survival", summary.get("test/episodic_survival")),
        "train_eval_survival": summary.get("train_eval/charts/episodic_survival", summary.get("train_eval/episodic_survival")),
        "legacy_survival": summary.get("charts/episodic_survival"),
    })

runs_df = pd.DataFrame(summary_rows)
print(f"Selected {len(selected_runs)} / {len(all_runs)} runs")
runs_df


## Download And Cache Histories

Each run is cached as a long-form CSV: `run_name`, `run_id`, `metric`, `step`, `value`. Re-run with `FORCE_REFRESH = True` when you want fresh W&B data.

In [ ]:
def safe_name(text):
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", str(text)).strip("._-")
    return text or "run"


def scan_metric(run, metric):
    rows = []
    for row in run.scan_history(keys=["_step", metric], page_size=PAGE_SIZE):
        if metric not in row:
            continue
        value = row.get(metric)
        if value is None or (isinstance(value, float) and np.isnan(value)):
            continue
        rows.append({
            "run_name": run.name,
            "run_id": run.id,
            "metric": metric,
            "step": row.get("_step"),
            "value": value,
        })
    return rows


def download_run_history(run, metrics):
    cache_path = CACHE_DIR / f"{safe_name(run.name)}__{run.id}.csv.gz"
    if cache_path.exists() and not FORCE_REFRESH:
        return pd.read_csv(cache_path)

    t0 = time.time()
    rows = []
    for metric in metrics:
        rows.extend(scan_metric(run, metric))
    history = pd.DataFrame(rows)
    if not history.empty:
        history = history.sort_values(["run_name", "metric", "step"])
    history.to_csv(cache_path, index=False)
    print(f"cached {run.name}: {len(history):,} rows in {time.time() - t0:.1f}s")
    return history


histories = []
for run in selected_runs:
    histories.append(download_run_history(run, METRICS))

history_df = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame(columns=["run_name", "run_id", "metric", "step", "value"])
history_df["step_millions"] = history_df["step"] / 1_000_000
print(history_df.shape)
history_df.head()


## Plot Helpers

In [ ]:
SURVIVAL_CANDIDATES = {
    "test": ["test/charts/episodic_survival", "test/episodic_survival", "charts/episodic_survival"],
    "train_eval": ["train_eval/charts/episodic_survival", "train_eval/episodic_survival"],
    "legacy": ["charts/episodic_survival"],
}


def _smooth(values, window):
    if window is None or window <= 1:
        return values
    return values.rolling(window=window, min_periods=1).mean()


def plot_metric(history, metric, run_regex=None, multiply=1.0, ylabel=None, title=None, smooth=1, ax=None):
    data = history[history["metric"] == metric].copy()
    if run_regex:
        data = data[data["run_name"].str.contains(run_regex, regex=True, na=False)]
    if data.empty:
        raise ValueError(f"No data found for metric {metric!r}")
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))
    for run_name, group in data.groupby("run_name", sort=False):
        group = group.sort_values("step")
        y = _smooth(group["value"].astype(float) * multiply, smooth)
        ax.plot(group["step_millions"], y, label=run_name, linewidth=2)
    ax.set_xlabel("Environment steps (millions)")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or metric)
    ax.legend(loc="best", fontsize=8)
    return ax


def plot_survival(history, split="test", run_regex=None, smooth=1, ax=None):
    candidates = SURVIVAL_CANDIDATES[split]
    rows = []
    for run_name, run_data in history.groupby("run_name", sort=False):
        if run_regex and not re.search(run_regex, run_name):
            continue
        chosen = None
        for metric in candidates:
            metric_data = run_data[run_data["metric"] == metric].copy()
            if not metric_data.empty:
                chosen = metric_data
                chosen["source_metric"] = metric
                break
        if chosen is not None:
            rows.append(chosen)
    if not rows:
        raise ValueError(f"No {split!r} survival data found")
    data = pd.concat(rows, ignore_index=True)
    if ax is None:
        _, ax = plt.subplots(figsize=(11, 5.5))
    for run_name, group in data.groupby("run_name", sort=False):
        group = group.sort_values("step")
        y = _smooth(group["value"].astype(float) * 100.0, smooth)
        ax.plot(group["step_millions"], y, label=run_name, linewidth=2)
    ax.set_xlabel("Environment steps (millions)")
    ax.set_ylabel("Survival (%)")
    ax.set_ylim(-2, 102)
    ax.set_title(f"{split} episodic survival")
    ax.legend(loc="best", fontsize=8)
    return ax


def save_current_figure(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    print(path)


## Survival Plots

In [ ]:
ax = plot_survival(history_df, split="test", smooth=1)
save_current_figure("test_survival.png")


In [ ]:
if (history_df["metric"].isin(SURVIVAL_CANDIDATES["train_eval"])).any():
    ax = plot_survival(history_df, split="train_eval", smooth=1)
    save_current_figure("train_eval_survival.png")
else:
    print("No train_eval survival metrics found for the selected runs.")


## Training Diagnostics

In [ ]:
for metric in ["train/entropy_coef", "train/action0_logit_bonus"]:
    if (history_df["metric"] == metric).any():
        plt.figure(figsize=(10, 5))
        plot_metric(history_df, metric, smooth=1)
        save_current_figure(safe_name(metric) + ".png")
        plt.show()

for metric in ["train/frac_action_0_agent_0", "train/frac_action_0_agent_1", "train/frac_action_0_agent_2"]:
    if (history_df["metric"] == metric).any():
        plt.figure(figsize=(10, 5))
        plot_metric(history_df, metric, ylabel="Fraction action 0", smooth=5)
        save_current_figure(safe_name(metric) + ".png")
        plt.show()


## Best And Latest Values

In [ ]:
def metric_summary(history, metric_candidates):
    rows = []
    for run_name, run_data in history.groupby("run_name", sort=False):
        chosen = None
        for metric in metric_candidates:
            data = run_data[run_data["metric"] == metric].sort_values("step")
            if not data.empty:
                chosen = data
                break
        if chosen is None:
            continue
        values = chosen["value"].astype(float)
        best_idx = values.idxmax()
        latest = chosen.iloc[-1]
        best = chosen.loc[best_idx]
        rows.append({
            "run_name": run_name,
            "metric": chosen.iloc[0]["metric"],
            "latest_step_m": latest["step"] / 1_000_000,
            "latest_survival_pct": latest["value"] * 100.0,
            "best_step_m": best["step"] / 1_000_000,
            "best_survival_pct": best["value"] * 100.0,
        })
    return pd.DataFrame(rows).sort_values("best_survival_pct", ascending=False)


test_summary = metric_summary(history_df, SURVIVAL_CANDIDATES["test"])
test_summary.to_csv(CACHE_DIR / "selected_runs_test_survival_summary.csv", index=False)
test_summary
